# TFIM hardware demo (Aer-first, hardware optional)

Runs fully on Aer without a token. Hardware cells skip cleanly if `QISKIT_IBM_TOKEN` is unset.
Colab: install pinned stack first, upload `hardware.py`.

In [ ]:
# Colab only — uncomment:
# %pip install "qiskit[visualization]==2.5.2" "qiskit-aer==0.17.2" "qiskit-ibm-runtime==0.50.0" "scipy==1.18.1" "matplotlib==3.11.2"

import os
import hardware as hw
print('hardware.py loaded')

## 1. Build TFIM circuit + observables

In [ ]:
qc = hw.build_tfim_circuit(n=4, j=1.0, h=0.5, dt=0.1, steps=3)
print(qc.count_ops(), 'depth=', qc.depth())
display(qc.draw('text'))
obs = hw.build_tfim_observables(n=4, j=1.0, h=0.5)
print('observables:', list(obs))

## 2. Exact baseline vs Aer

In [ ]:
exact = hw.exact_tfim_magnetization(n=4, j=1.0, h=0.5, dt=0.1, steps=3)
print('exact magnetization:', exact)
val = hw.run_expectation_aer(qc.remove_final_measurements(inplace=False) or qc, obs['magnetization'], shots=4096)
print('aer magnetization:', val, 'err:', abs(val - exact))

## 3. Transpilation comparison (Aer, no backend)

In [ ]:
rows = hw.transpile_compare(qc, backend=None)
rows

## 4. Manual ZNE on Aer

In [ ]:
core = qc.remove_final_measurements(inplace=False) or qc
zne = hw.run_with_zne_aer(core, obs['magnetization'], shots=4096)
print('mitigated:', zne['mitigated'], 'exact:', exact)

## 5. IBM hardware (needs QISKIT_IBM_TOKEN in .env or env var)

In [ ]:
svc = hw.get_service()
if svc is None:
    print('offline — skipping hardware (expected without token)')
else:
    backend = hw.select_backend(svc)
    info = hw.backend_properties_dict(backend)
    lvl = hw.transpile_compare(core, backend)
    aware, cmp = hw.hardware_aware_transpile(core, backend, n=4)
    r0 = hw.run_expectation_hardware(core, obs['magnetization'], backend, shots=4096, resilience_level=0)
    r2 = hw.run_expectation_hardware(core, obs['magnetization'], backend, shots=4096, use_zne=True)
    print('no-mit:', r0['value'], 'zne:', r2['value'], 'exact:', exact)

## 6. Save JSON + plots (Aer-only analysis)

In [ ]:
from qiskit import transpile
aer_vals = {}
for r in rows:
    lv = r['optimization_level']
    t = transpile(core, optimization_level=lv, seed_transpiler=42)
    aer_vals[str(lv)] = hw.run_expectation_aer(t, obs['magnetization'], shots=4096)
analysis = hw.analyze(rows, exact, aer_vals, {'mitigated': zne['mitigated'], 'raw': zne['raw']})
hw.save_json({'analysis': analysis}, 'results/demo_results.json')
hw.make_plots(analysis, 'results')
print('done → results/')